# Prompt caching

Cache a stable input prefix, not the generated answer. This notebook covers the
rules and the practical setup: mark the last tool and the system text, repeat a
request, then change each component and inspect token usage.

## Rules

- Processing order: **tools → system → messages**. A breakpoint includes its own block.
- The prefix must match exactly; questions after the breakpoint may change.
- Explicit caching supports up to four breakpoints. Default TTL is 5 minutes;
  a hit refreshes it. `ttl="1h"` is an optional, more expensive write.
- Minimum prefix length depends on the model: Sonnet 4.5 needs 1024 tokens,
  Haiku 4.5 needs 4096. Short prefixes are accepted without being cached.
- Cache writes cost more than ordinary input; repeated reads are cheaper.
  Decide whether it helps from usage, not latency alone.

The lessons use explicit breakpoints. The current API also offers top-level
`cache_control` for automatic placement; this notebook keeps explicit placement
so the boundaries are visible.

## Setup

Use Sonnet 4.5 to let the reminder schemas reach the smaller minimum on their own.
The tool definitions come from module 4; this notebook never executes them.
The long system context is the report from module 5.

In [1]:
from copy import deepcopy
from pathlib import Path
from time import perf_counter
import sys
from uuid import uuid4

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

sys.path.insert(0, str(Path("../04_tool_use").resolve()))
from tools import (
    add_duration_to_datetime_schema,
    get_current_datetime_schema,
    set_reminder_schema,
)

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-5"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def request_params(messages, system=None, tools=None, cache=True):
    params = {
        "model": model,
        "max_tokens": 120,
        "temperature": 0.0,
        "messages": messages,
    }
    if tools:
        tools_clone = tools.copy()
        if cache:
            last_tool = tools_clone[-1].copy()
            last_tool["cache_control"] = {"type": "ephemeral"}
            tools_clone[-1] = last_tool
        params["tools"] = tools_clone
        # Only study prompt processing, without tool execution.
        params["tool_choice"] = {"type": "none"}
    if system:
        params["system"] = [{"type": "text", "text": system}]
        if cache:
            params["system"][-1]["cache_control"] = {"type": "ephemeral"}
    return params


def chat(messages, system=None, tools=None, cache=True):
    return client.messages.create(**request_params(messages, system, tools, cache))


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


results = []


def run_case(label, question, system, tools=None):
    messages = []
    add_user_message(messages, question)
    start = perf_counter()
    response = chat(messages, system=system, tools=tools)
    usage = response.usage
    row = {
        "case": label,
        "input": usage.input_tokens,
        "write": usage.cache_creation_input_tokens or 0,
        "read": usage.cache_read_input_tokens or 0,
        "output": usage.output_tokens,
        "seconds": round(perf_counter() - start, 2),
        "stop_reason": response.stop_reason,
        "cost": getattr(usage, "cost", None),
    }
    results.append(row)
    print(text_from_message(response))
    print(row)
    return response

## Below the minimum

A breakpoint on a tiny system prompt is valid, but cannot establish a cache.
No tools or long context are included in this request.

In [2]:
short_response = run_case(
    "short prefix", "What is 1 + 1? Answer in one short sentence.",
    system="Answer arithmetic questions briefly.",
)

1 + 1 equals 2.
{'case': 'short prefix', 'input': 27, 'write': 0, 'read': 0, 'output': 14, 'seconds': 2.01, 'stop_reason': 'end_turn', 'cost': 0.000291}


## Add two breakpoints without changing the source tools

The first breakpoint ends at the last tool. The second includes the system text
and everything before it. Copy the tools list and its last dictionary before
adding a marker; the original definitions stay reusable.

A fresh run ID at the beginning of the first tool prevents reuse of this
experiment's previous run. Keep it unchanged across the following requests.

In [3]:
tools = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
]
original_tools = deepcopy(tools)
run_id = uuid4().hex
experiment_tools = deepcopy(tools)
experiment_tools[0]["description"] = f"Experiment {run_id}. " + experiment_tools[0]["description"]

tools_before_request = deepcopy(experiment_tools)
report = Path("../05_rag/report.md").read_text()
system_prompt = (
    "You answer questions about this annual report. Use only the report, "
    "answer in one short sentence, and do not call tools.\n\n" + report
)

preview = request_params(
    [{"role": "user", "content": "Summarize the report."}],
    system=system_prompt, tools=experiment_tools,
)
assert "cache_control" not in tools[-1]
assert "cache_control" not in experiment_tools[-1]
assert preview["tools"][-1]["cache_control"] == {"type": "ephemeral"}
assert preview["system"][0]["cache_control"] == {"type": "ephemeral"}
print("Two explicit breakpoints; source tool definitions unchanged.")

Two explicit breakpoints; source tool definitions unchanged.


## First request and a changed question

The first call should write the eligible prefixes. The next question is after
both breakpoints, so the same tools and system context can be read from cache.
Calls run sequentially, within the default TTL.

In [4]:
first_response = run_case(
    "initial prefix", "What percentage reduction in critical failures did Software Engineering measure? Answer with the percentage only.",
    system_prompt, experiment_tools,
)
repeat_response = run_case(
    "new question", "What was the year-over-year group revenue growth? Answer with the percentage only.",
    system_prompt, experiment_tools,
)

40%
{'case': 'initial prefix', 'input': 339, 'write': 5050, 'read': 0, 'output': 6, 'seconds': 2.59, 'stop_reason': 'end_turn', 'cost': 0.0200445}


3.1%
{'case': 'new question', 'input': 340, 'write': 0, 'read': 5050, 'output': 8, 'seconds': 2.04, 'stop_reason': 'end_turn', 'cost': 0.002655}


## Change the system, then change a tool

Changing the start of the system text should keep the tools prefix reusable,
but write a new system prefix. This partial hit needs the tools breakpoint to
meet the model's minimum by itself.

Changing the first tool invalidates both prefixes because tools come first.
Repeat the changed request to check that the new prefix is reusable.

In [5]:
changed_system = "Updated answer style: be concise.\n" + system_prompt
system_response = run_case(
    "changed system", "What percentage reduction in critical failures did Software Engineering measure? Answer with the percentage only.",
    changed_system, experiment_tools,
)

changed_tools = deepcopy(experiment_tools)
changed_tools[0]["description"] = "Revised tool guidance. " + changed_tools[0]["description"]
tool_response = run_case(
    "changed tool", "What percentage reduction in critical failures did Software Engineering measure? Answer with the percentage only.",
    changed_system, changed_tools,
)
changed_repeat_response = run_case(
    "repeat changed prefix", "What was the year-over-year group revenue growth? Answer with the percentage only.",
    changed_system, changed_tools,
)

40%
{'case': 'changed system', 'input': 339, 'write': 3959, 'read': 1100, 'output': 6, 'seconds': 2.4, 'stop_reason': 'end_turn', 'cost': 0.01628325}


40%
{'case': 'changed tool', 'input': 339, 'write': 5064, 'read': 0, 'output': 6, 'seconds': 2.44, 'stop_reason': 'end_turn', 'cost': 0.020097}


3.1%
{'case': 'repeat changed prefix', 'input': 340, 'write': 0, 'read': 5064, 'output': 8, 'seconds': 2.22, 'stop_reason': 'end_turn', 'cost': 0.0026592}


## Inspect the measurements

`cache_creation_input_tokens` measures writes; `cache_read_input_tokens` measures
reads. `input_tokens` is uncached input in the Anthropic usage format. Cost, when
returned by the provider, is the full request cost, including generated output.
Elapsed time is noisy and is not an assertion about speedup.

Check behavior explicitly instead of treating a successful answer as a cache hit.

In [6]:
print(f"{'case':<23} {'input':>7} {'write':>7} {'read':>7} {'output':>7} {'seconds':>8} {'cost ($)':>10}")
for row in results:
    cost = f"{row['cost']:.6f}" if row["cost"] is not None else "n/a"
    print(f"{row['case']:<23} {row['input']:>7} {row['write']:>7} "
          f"{row['read']:>7} {row['output']:>7} {row['seconds']:>8.2f} {cost:>10}")

short, first, repeat, changed_sys, changed_tool, changed_repeat = results
checks = {
    "Short prefix was not cached": short["write"] == short["read"] == 0,
    "Initial prefix was written": first["write"] > 0,
    "New question reused the prefix": repeat["read"] > 0 and repeat["write"] == 0,
    "Changed system read tools and wrote a new prefix": changed_sys["read"] > 0 and changed_sys["write"] > 0,
    "Changed first tool invalidated both prefixes": changed_tool["read"] == 0 and changed_tool["write"] > 0,
    "Changed prefix was reusable": changed_repeat["read"] > 0 and changed_repeat["write"] == 0,
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'NOT OBSERVED'}: {label}")

assert all(row["stop_reason"] == "end_turn" for row in results)
assert tools == original_tools
assert experiment_tools == tools_before_request
print("Source tool definitions and experiment inputs remain unchanged.")

case                      input   write    read  output  seconds   cost ($)
short prefix                 27       0       0      14     2.01   0.000291
initial prefix              339    5050       0       6     2.59   0.020044
new question                340       0    5050       8     2.04   0.002655
changed system              339    3959    1100       6     2.40   0.016283
changed tool                339    5064       0       6     2.44   0.020097
repeat changed prefix       340       0    5064       8     2.22   0.002659
PASS: Short prefix was not cached
PASS: Initial prefix was written
PASS: New question reused the prefix
PASS: Changed system read tools and wrote a new prefix
PASS: Changed first tool invalidated both prefixes
PASS: Changed prefix was reusable
Source tool definitions and experiment inputs remain unchanged.


These measurements apply to the selected model and provider route for this run.
If a check is not observed, inspect prefix size, TTL, request differences and
provider routing before assigning a cause. The experiment does not wait for TTL
expiry or test the one-hour option.

Next lesson: code execution and the Files API.